# 03 — The Perceptron: A Single Artificial Neuron

## Biological inspiration

A real neuron receives signals from many neighbours, adds them up,  
and *fires* (sends a signal onward) only when the total is strong enough.

An artificial neuron does the same thing mathematically:

```
z    = w₁·x₁ + w₂·x₂ + ... + b      ← weighted sum + bias
output = sigmoid(z)                    ← squash to a probability (0 – 1)
```

The **weights** `w` say how much each input matters.  
The **bias** `b` shifts the threshold.

We'll train a single neuron to learn two simple logic gates: **OR** and **AND**.

## The Sigmoid Activation Function

The sigmoid squashes any number into the range (0, 1):

```
sigmoid(z) = 1 / (1 + e^{-z})
```

- Very negative z → output near 0  (neuron is *off*)
- Near-zero z     → output near 0.5  (uncertain)
- Very positive z → output near 1  (neuron is *on*)

Let's plot it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

z_vals = np.linspace(-6, 6, 200)

plt.figure(figsize=(8, 3))
plt.plot(z_vals, sigmoid(z_vals), color="darkorange", linewidth=2, label="sigmoid(z)")
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("z  (weighted sum)")
plt.ylabel("output")
plt.title("Sigmoid function — maps any number to (0, 1)")
plt.legend()
plt.tight_layout()
plt.show()

## The OR Gate Dataset

OR is true when *at least one* input is 1:

| A | B | A OR B |
|---|---|--------|
| 0 | 0 |   0    |
| 0 | 1 |   1    |
| 1 | 0 |   1    |
| 1 | 1 |   1    |

These 4 rows are our entire training set.

In [ ]:
# Each row is one training example: [input_A, input_B]
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

y_or = np.array([0, 1, 1, 1])   # expected output for OR

# Visualise the four points
plt.figure(figsize=(4, 4))
for (a, b), label in zip(X, y_or):
    color = "green" if label == 1 else "red"
    marker = "o" if label == 1 else "s"
    plt.scatter(a, b, c=color, marker=marker, s=120, zorder=3)

# Legend proxies
import matplotlib.patches as mpatches
plt.legend(handles=[mpatches.Patch(color="green", label="OR = 1"),
                     mpatches.Patch(color="red",   label="OR = 0")])
plt.xlim(-0.3, 1.3); plt.ylim(-0.3, 1.3)
plt.xticks([0, 1]); plt.yticks([0, 1])
plt.xlabel("Input A"); plt.ylabel("Input B")
plt.title("OR gate — can a straight line separate the classes?")
plt.tight_layout()
plt.show()

## Matrix Notation: Processing All Examples at Once

For **one** example the weighted sum is simply:
```python
z = w[0] * input_A + w[1] * input_B + b
```

But we have **4 examples**.  Instead of looping, we use matrix multiplication:
```python
# X has shape (4, 2) — 4 rows, 2 columns (one column per feature)
# w has shape (2,)   — one weight per feature
# X @ w computes the dot product for every row at once
z = X @ w + b    # result shape: (4,) — one z-value per example
```

Think of it as doing the single-example formula 4 times simultaneously.

## Training the Perceptron on OR

**Cross-entropy loss** measures how far our probability predictions are  
from the true 0/1 labels — it penalises confident wrong answers heavily.

**Gradient for the weights:**  
Each weight `w[j]` should change by how much it contributed to the error,  
averaged across all training examples:
```
grad_w = (1/n) * Xᵀ · error
```
Where `error = predicted - actual`.  `Xᵀ` (X transposed) pairs each  
feature column with the corresponding errors across examples.

In [ ]:
np.random.seed(0)
w = np.random.randn(2) * 0.1   # small random starting weights
b = 0.0

learning_rate = 0.5
loss_history_or = []

for step in range(1000):
    # --- Forward pass ---
    z = X @ w + b              # weighted sum for all 4 examples at once
    y_hat = sigmoid(z)         # predicted probabilities

    # --- Cross-entropy loss ---
    loss = -np.mean(
        y_or * np.log(y_hat + 1e-9) + (1 - y_or) * np.log(1 - y_hat + 1e-9)
    )
    loss_history_or.append(loss)

    # --- Backward pass (gradients) ---
    error = y_hat - y_or                   # how wrong are we per example?
    grad_w = X.T @ error / len(X)          # weight gradient: error × feature, averaged
    grad_b = error.mean()                  # bias gradient: plain average of errors

    # --- Update ---
    w -= learning_rate * grad_w
    b -= learning_rate * grad_b

print('Trained! OR predictions:')
for inputs, expected in zip(X, y_or):
    pred = sigmoid(inputs @ w + b)
    print(f'  {inputs[0]} OR {inputs[1]}  → {pred:.3f}  (expected {expected})')

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(loss_history_or, color="darkorange")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("OR perceptron — loss during training")
plt.tight_layout()
plt.show()

## Visualising the Decision Boundary

A trained perceptron draws a **straight line** (in 2-D) that separates  
class 0 from class 1.  Let's plot it.

In [ ]:
# Background colour map
xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 200),
                     np.linspace(-0.3, 1.3, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
probs = sigmoid(grid @ w + b).reshape(xx.shape)

plt.figure(figsize=(5, 5))
plt.contourf(xx, yy, probs, levels=50, cmap="RdYlGn", alpha=0.7)
plt.colorbar(label="P(OR = 1)")

for (a, b_pt), label in zip(X, y_or):
    color = "green" if label == 1 else "red"
    marker = "o" if label == 1 else "s"
    plt.scatter(a, b_pt, c=color, marker=marker, s=150,
               edgecolors="black", linewidths=1.5, zorder=5)

plt.xlim(-0.3, 1.3); plt.ylim(-0.3, 1.3)
plt.xticks([0, 1]); plt.yticks([0, 1])
plt.xlabel("Input A"); plt.ylabel("Input B")
plt.title("Decision boundary for OR")
plt.tight_layout()
plt.show()

## The AND Gate

AND is true only when **both** inputs are 1.  Let's train quickly.

In [ ]:
y_and = np.array([0, 0, 0, 1])

np.random.seed(0)
w2 = np.random.randn(2) * 0.1
b2 = 0.0

for _ in range(1000):
    z = X @ w2 + b2
    y_hat = sigmoid(z)
    error = y_hat - y_and
    w2 -= learning_rate * (X.T @ error / len(X))
    b2 -= learning_rate * error.mean()

print('AND predictions:')
for inputs, expected in zip(X, y_and):
    pred = sigmoid(inputs @ w2 + b2)
    print(f'  {inputs[0]} AND {inputs[1]}  → {pred:.3f}  (expected {expected})')

## The XOR Challenge — Where a Single Neuron Fails

XOR is 1 when **exactly one** input is 1:

| A | B | A XOR B |
|---|---|---------|
| 0 | 0 |    0    |
| 0 | 1 |    1    |
| 1 | 0 |    1    |
| 1 | 1 |    0    |

A single neuron can only draw a **straight line** to separate classes.  
But XOR is not linearly separable — no single line can separate  
the two groups below. That's why we need **multiple layers** (Chapter 04).

In [ ]:
y_xor = np.array([0, 1, 1, 0])

# Train anyway and see the best it can do
np.random.seed(0)
w3 = np.random.randn(2) * 0.1
b3 = 0.0
for _ in range(5000):
    z = X @ w3 + b3
    y_hat = sigmoid(z)
    error = y_hat - y_xor
    w3 -= learning_rate * (X.T @ error / len(X))
    b3 -= learning_rate * error.mean()

print('XOR predictions (single neuron — watch it struggle):')
for inputs, expected in zip(X, y_xor):
    pred = sigmoid(inputs @ w3 + b3)
    print(f'  {inputs[0]} XOR {inputs[1]}  → {pred:.3f}  (expected {expected})')

# Show why: plot the decision boundary
probs_xor = sigmoid(grid @ w3 + b3).reshape(xx.shape)
plt.figure(figsize=(5, 5))
plt.contourf(xx, yy, probs_xor, levels=50, cmap="RdYlGn", alpha=0.7)
plt.colorbar(label="P(XOR = 1)")
for (a, bpt), label in zip(X, y_xor):
    color = "green" if label == 1 else "red"
    marker = "o" if label == 1 else "s"
    plt.scatter(a, bpt, c=color, marker=marker, s=150,
               edgecolors="black", linewidths=1.5, zorder=5)
plt.xlim(-0.3, 1.3); plt.ylim(-0.3, 1.3)
plt.xticks([0, 1]); plt.yticks([0, 1])
plt.xlabel("Input A"); plt.ylabel("Input B")
plt.title("XOR — no straight line can separate the two classes!")
plt.tight_layout()
plt.show()

## Key Lesson

- A perceptron can learn **linearly separable** problems (OR, AND) perfectly.
- It fails on **non-linear** problems like XOR.
- The fix: stack multiple neurons in **layers** → a neural network (next notebook!).